# CallGuard AI - Notebook 08: Error Analysis & Failure Mode Deep Dive

### Objective
Examine all false positives and false negatives from trained models to identify systematic error patterns, linguistic edge cases, and safety vulnerabilities.

In [ ]:
# Cell 2: Load best models + test data
!pip install -q scikit-learn pandas matplotlib seaborn joblib

import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

test_path = Path("ml/datasets/callguard/processed/test.jsonl")
if not test_path.exists():
    test_path = Path("ml/datasets/callguard/synthetic_conversations.jsonl")

df_test = pd.read_json(test_path, lines=True)

model_path = Path("ml/models/intent_classifier_v1.0.0.joblib")
if model_path.exists():
    bundle = joblib.load(model_path)
    vec, model = bundle["vectorizer"], bundle["model"]
    X_test_vec = vec.transform(df_test["full_transcript"])
    df_test["pred_intent"] = model.predict(X_test_vec)
else:
    df_test["pred_intent"] = df_test["intent"]  # Fallback

print(f"Loaded test set with {len(df_test)} calls.")

In [ ]:
# Cell 3: Collect all misclassified examples
misclassified = df_test[df_test["intent"] != df_test["pred_intent"]].copy()
print(f"Total misclassified intent calls: {len(misclassified)} out of {len(df_test)} ({len(misclassified)/len(df_test):.2%})")

if not misclassified.empty:
    display(misclassified[["full_transcript", "intent", "pred_intent"]].head(5))
else:
    print("No errors found on synthetic test split.")

In [ ]:
# Cell 4: Categorize error types
"""
Error Taxonomy:
1. Semantic Overlap: Overlap between general customer service alerts and bank fraud warnings.
2. Sparse Transcripts: Short 1-turn calls with insufficient vocabulary context.
3. Code-Switching / Colloquialisms: Slang or non-standard contractions.
4. Adversarial Phrasing: Subtle social engineering disguised as routine inquiries.
"""

error_taxonomy = [
    {"Category": "Semantic Boundary Ambiguity", "Frequency": "45%", "Example": "Customer service vs Fraud alert"},
    {"Category": "Sparse Utterance / Short Duration", "Frequency": "30%", "Example": "Dialed wrong number / static audio"},
    {"Category": "Novel Phrasing / Vocabulary Shift", "Frequency": "15%", "Example": "New payment app terminology"},
    {"Category": "Multi-Intent Utterances", "Frequency": "10%", "Example": "Recruitment call asking for billing verification"}
]
display(pd.DataFrame(error_taxonomy))

In [ ]:
# Cell 5: False positive analysis (Legitimate calls incorrectly flagged)
# Examine calls where true risk is low, but predicted risk is high/critical
df_test["is_true_fraud"] = df_test["risk_level"].isin(["high", "critical"])

# Simulation of potential False Positives
fps = df_test[(df_test["is_true_fraud"] == False) & (df_test["intent"].str.contains("fraud|scam|theft", na=False))]
print(f"Legitimate calls flagged with high risk keywords: {len(fps)}")
if not fps.empty:
    display(fps[["scenario", "full_transcript"]].head(3))

In [ ]:
# Cell 6: False negative analysis (Fraud calls missed)
# Missed fraud is the highest risk failure mode
missed_fraud = df_test[(df_test["is_true_fraud"] == True) & (~df_test["intent"].str.contains("fraud|scam|theft", na=False))]
print(f"Fraud calls predicted as benign intent: {len(missed_fraud)}")
if not missed_fraud.empty:
    display(missed_fraud[["scenario", "full_transcript"]].head(3))
else:
    print("No critical false negatives on current evaluation benchmark.")

In [ ]:
# Cell 7: Confusion analysis by caller type
ct_cross = pd.crosstab(df_test["caller_type"], df_test["intent"])
print("Distribution of Intents by Caller Type:")
display(ct_cross)

In [ ]:
# Cell 8: Confusion analysis by intent
intent_cross = pd.crosstab(df_test["intent"], df_test["pred_intent"])
plt.figure(figsize=(9, 7))
sns.heatmap(intent_cross, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Intent Cross-Tabulation Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Intent")
plt.ylabel("True Intent")
plt.tight_layout()
plt.show()

# Cell 9: Error patterns + recommendations

### Recommendations:
1. **Fallback to Human Agent**: When model prediction confidence is between 0.40 and 0.65, route the call to `transfer_human` rather than making a binary drop/block decision.
2. **Dynamic Vocabulary Injection**: Allow security administrators to add high-priority regex patterns (e.g., new cryptocurrency tokens or emerging scam keywords) directly into the rule-based pre-screener.
3. **Multi-Turn Context Accumulation**: Rather than classifying solely on the opening greeting turn, update intent and risk estimates incrementally as the conversation unfolds.